# 现代 LLM 推理系统：从一个请求到一张 GPU 上的 100 个请求

> 20–23 主要在优化单个请求。真正的 vLLM / SGLang / TensorRT-LLM 服务面对的是另一个问题：**很多长度不同、到达时间不同的请求，怎么一起跑？**
>
> 这一章的目标就是让你无压力看懂厂商系统报告和招聘 JD 里的核心名词：
>
> `Throughput / TTFT / TPOT / Continuous Batching / PagedAttention / Prefix Caching / Chunked Prefill / PD Disaggregation / RadixAttention`

先定义目标，再看系统为什么需要这些机制。


## 1. 服务指标：不要只说“延迟”

- **Throughput**：整个系统每秒处理 / 生成多少 Token
- **TTFT**：请求到第一个 Token 的时间
- **TPOT / ITL**：后续 Token 之间的平均间隔
- **Concurrency**：同时承载多少请求

Prefill 更影响 TTFT，Decode 更影响 TPOT。提高并发通常能提高吞吐，但也可能让单请求延迟变差。


## 2. 一个很实用的 Roofline 直觉：Decode 为什么常常吃带宽

7B BF16 权重约 14 GB。decoder-only 模型一次 token forward 的矩阵乘粗略可按：

$$
\mathrm{FLOPs/token} \approx 2 \times \#params
$$

这是教学近似，不包含所有 attention / embedding / MoE 细节。


In [ ]:
params = 7e9
gpu_tflops = 312
gpu_bw_gbs = 2000

flops_per_token = 2 * params
compute_tokens_s = gpu_tflops*1e12/flops_per_token

weight_bytes = params*2
bandwidth_single_stream = gpu_bw_gbs*1e9/weight_bytes

print("compute roofline proxy:", round(compute_tokens_s), "token/s")
print("single-stream weight-bandwidth proxy:", round(bandwidth_single_stream), "token/s")
print("注意：这只是解释为何 decode 容易 memory-bound 的数量级直觉。")


## 3. Static Batching 为什么浪费？

静态 Batch 的问题不是“最长请求决定所有请求最终完成时间”这么简单，而是：

- Batch 形成后，新请求进不来
- 某些序列提前结束，slot 可能空着
- 不同序列长度导致 padding / 空闲

Continuous Batching 的核心是 **iteration-level scheduling**：每个 Decode iteration 都可以让已完成请求退出、新请求补进来。


In [ ]:
def simulate_static(lengths, batch_size):
    t=0
    for i in range(0,len(lengths),batch_size):
        t += max(lengths[i:i+batch_size])
    return t

def simulate_continuous(lengths, batch_size):
    pending=list(lengths)
    active=[]
    t=0
    while pending or active:
        while pending and len(active)<batch_size:
            active.append(pending.pop(0))
        active=[x-1 for x in active]
        active=[x for x in active if x>0]
        t+=1
    return t

req=[5,200,8,150,6,180,10,120]
print("static steps:", simulate_static(req,4))
print("continuous steps:", simulate_continuous(req,4))


## 4. PagedAttention：KV Cache 为什么要“分页”？

连续 KV Cache 分配很容易出现：

- 预留太多 -> 浪费
- 序列长度变化 -> 碎片
- 请求结束 -> 空洞

PagedAttention 把 KV Cache 切成固定大小的 block/page，通过映射表管理逻辑连续、物理离散的缓存。

把它类比成操作系统虚拟内存比“某个 Attention 新算法”更准确：**Attention 公式没变，主要变化是 KV Cache 的内存管理。**


## 5. Prefix Caching / RadixAttention：相同前缀为什么还要重复 Prefill？

很多请求共享前缀：

```text
同一个 system prompt
同一份长文档
同一段多轮对话历史
```

Prefix Cache 直接复用已经算好的前缀 KV。

SGLang 报告里常见的 **RadixAttention**，可以把共享前缀组织成 radix tree，让不同请求复用公共 KV 前缀。

看到这些词时先记住：它们主要减少的是 **重复 Prefill**。


## 6. Chunked Prefill：长 Prompt 为什么会“堵车”？

一个超长 Prefill 如果一次吃完整 GPU，短 Decode 请求可能等很久，TPOT 变差。

Chunked Prefill 把长 Prompt 拆成多个 chunk，让 scheduler 能在 Prefill 和 Decode 之间更细粒度地穿插工作。

它解决的是：

> **长 Prefill 抢占资源，如何避免把在线 Decode 饿死？**


## 7. PD 分离：厂商最爱讲的 Prefill / Decode Disaggregation

Prefill 和 Decode 的硬件需求不同：

```text
Prefill: 大矩阵、并行度高、偏 compute-bound
Decode : 小矩阵、频繁读权重/KV、偏 memory-bound
```

**PD 分离（Prefill-Decode Disaggregation）** 就是把两个阶段放到不同 worker / GPU 池，分别优化和扩缩容，再通过 KV Cache 传输把请求接起来。

它不是“把 Transformer 拆成两半”，而是把**同一个请求的两个阶段做资源解耦**。


## 8. 一张表读懂厂商“黑话”

| 名词 | 先问：它主要在解决什么？ |
|---|---|
| Continuous Batching | GPU slot 空闲和动态请求调度 |
| PagedAttention | KV Cache 碎片与动态分配 |
| Prefix Caching | 重复 Prefill |
| RadixAttention | 结构化共享前缀 KV |
| Chunked Prefill | 长 Prompt 阻塞 Decode |
| PD 分离 | Prefill / Decode 资源特征不同 |
| Tensor Parallel | 单模型太大，一张卡放不下 / 算不完 |
| Pipeline Parallel | 层级跨设备流水 |
| Expert Parallel | MoE experts 跨设备分布 |
| KV Cache quantization | KV Cache 太大 |
| Speculative Decoding | 串行 Target step 太多 |


## 9. 面试 / 招聘 JD 怎么把这些词串起来？

遇到“熟悉 vLLM / SGLang 推理优化”时，不要只背 API。你至少应该能讲出：

```text
请求进来
 -> scheduler
 -> prefill / decode
 -> continuous batching
 -> KV cache allocation (paged)
 -> prefix reuse
 -> long-prefill scheduling
 -> decode loop
 -> metrics: TTFT / TPOT / throughput
```

下一章回答另一个现实问题：

> **这些优化把系统跑快了，怎么证明模型质量没有掉、对比是公平的？**
